# EDA - full cross features (`build_full_cross_features`)

Validates the prior-ranked tier-A 12 cross-table features built across
**all five contest tables** (main, runs, shots, pressure, passes).

| # | Feature (logical) | Mechanism |
|---|-------------------|-----------|
| 1 | `four_domain_top_third_avg`               | mean of 4 spatial shares (run + press + pass + shots) |
| 2 | `top_third_consistency_count`             | count of domains with share > 0.3 |
| 3 | `top_third_pass_x_top_third_run`          | passing AND running in attacking third |
| 4 | `cumul_passes_{A,M,D,G}`                  | recovers position-confounded negative pooled coef |
| 5 | `passes_received_share_{A,M,D,G}`         | target-by-role indicator |
| 6 | `shots_per_pass_received`                 | clean target-striker proxy |
| 7 | `composure_under_pressure_ratio`          | (1 - press_turnover_rate) / pass_accuracy |
| 8 | `last15_passes_received_x_last15_sprints` | breakaway target signal |
| 9 | `passes_received_share_above_team_avg`    | team's main reception target |
| 10 | `subbed_x_last15_intensity_shots`        | impact-sub signal |
| 11 | `shot_to_pass_ratio`                     | striker vs distributor fingerprint |
| 12 | `shot_to_press_ratio`                    | clinical conversion under pressure |

The 12 logical features expand to **18 columns** because #4 and #5 each
produce 4 position-dummies. Plus the 2 ID columns -> 20 columns total.

Section structure mirrors the prior cross-EDA notebooks:

| ID | Section |
|----|---------|
| A | Build & profile |
| B | Bivariate signal vs `scored_after` |
| C | Redundancy among the 18 columns |
| D | Cluster-robust GLM + BH-FDR |
| E | Final manifest |


## Setup

In [1]:
"""Imports."""
from __future__ import annotations
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT: Path = Path.cwd().parent if Path.cwd().name == "eda" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from features.cross import build_full_cross_features, FULL_CROSS_FEATURE_COLUMNS


In [2]:
"""Display options."""
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 80)
pd.set_option("display.precision", 4)


In [3]:
"""Load all five source CSVs."""
DATA_DIR = PROJECT_ROOT / "data"
main   = pd.read_csv(DATA_DIR / "players_quarters_final.csv")
runs   = pd.read_csv(DATA_DIR / "player_appearance_run.csv")
shots  = pd.read_csv(DATA_DIR / "player_appearance_shot_limited.csv")
press  = pd.read_csv(DATA_DIR / "player_appearance_behaviour_under_pressure.csv")
passes = pd.read_csv(DATA_DIR / "player_appearance_pass.csv")
main.shape, runs.shape, shots.shape, press.shape, passes.shape


((3486, 33), (35133, 10), (780, 12), (12185, 10), (29795, 7))

In [4]:
"""Helpers (Wilson CI + binned-rate table)."""
from scipy import stats as _stats


def wilson_ci(successes: int, trials: int, alpha: float = 0.05) -> tuple[float, float]:
    if trials == 0:
        return float("nan"), float("nan")
    p = successes / trials
    z = _stats.norm.ppf(1.0 - alpha / 2.0)
    denom = 1.0 + (z ** 2) / trials
    centre = (p + (z ** 2) / (2.0 * trials)) / denom
    half = z * np.sqrt(p * (1.0 - p) / trials + (z ** 2) / (4.0 * trials ** 2)) / denom
    return float(centre - half), float(centre + half)


def binned_rate(df: pd.DataFrame, feature: str, target: str = "scored_after",
                bins=None, by: str | None = None) -> pd.DataFrame:
    work = df[[feature, target] + ([by] if by else [])].copy()
    if bins is not None:
        work["__bin"] = pd.cut(work[feature], bins=bins, include_lowest=True)
    else:
        work["__bin"] = work[feature]
    grouping = ["__bin"] if by is None else [by, "__bin"]
    g = work.groupby(grouping, observed=False, dropna=False)[target].agg(["size", "sum"])
    g = g.rename(columns={"size": "n", "sum": "n_pos"}).reset_index()
    g["rate"] = g["n_pos"] / g["n"].where(g["n"] > 0)
    ci = g.apply(lambda r: wilson_ci(int(r["n_pos"]), int(r["n"])), axis=1, result_type="expand")
    g["ci_lo"], g["ci_hi"] = ci[0], ci[1]
    return g.rename(columns={"__bin": "bin"})


## Section A - Build & profile

In [5]:
"""Build the full-cross feature frame."""
cross = build_full_cross_features(main, runs, shots, press, passes)
print(f"shape: {cross.shape}")
assert cross.shape[0] == len(main), "row count must match the main panel"
assert list(cross.columns) == list(FULL_CROSS_FEATURE_COLUMNS)
cross.head(3)


shape: (3486, 20)


,player_appearance_id,checkpoint,four_domain_top_third_avg,top_third_consistency_count,top_third_pass_x_top_third_run,cumul_passes_A,cumul_passes_M,cumul_passes_D,cumul_passes_G,passes_received_share_A,passes_received_share_M,passes_received_share_D,passes_received_share_G,shots_per_pass_received,composure_under_pressure_ratio,last15_passes_received_x_last15_sprints,passes_received_share_above_team_avg,subbed_x_last15_intensity_shots,shot_to_pass_ratio,shot_to_press_ratio
0,39421,H1_15,0.0,0,NaN,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.6667,0.0,NaN,0.0,0.0509,0.0,0.0,NaN
1,39422,H1_15,0.0,0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.5,0.0000,0.0,0.7,14.0,-0.1158,0.0,0.0,0.0
2,39423,H1_15,0.0,0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.5,0.0000,0.0,NaN,14.0,-0.1158,0.0,0.0,NaN


In [6]:
"""Univariate description."""
NUMERIC = [c for c in cross.columns if c not in ("player_appearance_id", "checkpoint")]
desc = cross[NUMERIC].describe().T[["count", "mean", "std", "min", "50%", "max"]]
desc["n_nan"] = len(cross) - desc["count"].astype(int)
desc.round(4)


,count,mean,std,min,50%,max,n_nan
four_domain_top_third_avg,3482.0,0.2542,0.2198,0.0000,0.2035,1.0000,4
top_third_consistency_count,3486.0,1.3652,1.4171,0.0000,1.0000,4.0000,0
top_third_pass_x_top_third_run,3312.0,0.0902,0.1267,0.0000,0.0397,1.0000,174
cumul_passes_A,3486.0,1.7344,5.8306,0.0000,0.0000,80.0000,0
cumul_passes_M,3486.0,7.0014,13.3630,0.0000,0.0000,111.0000,0
cumul_passes_D,3486.0,10.5433,18.1970,0.0000,0.0000,109.0000,0
cumul_passes_G,3486.0,1.5353,5.8690,0.0000,0.0000,78.0000,0
passes_received_share_A,3472.0,0.0973,0.2178,0.0000,0.0000,1.0000,14
passes_received_share_M,3475.0,0.1893,0.2510,0.0000,0.0000,1.0000,11
passes_received_share_D,3482.0,0.1762,0.2382,0.0000,0.0000,1.0000,4


### A - findings (interpretation)

Sanity-check expectations:

* `four_domain_top_third_avg` bounded in `[0, 1]`. Median around 0.2
  reflects most players' average attacking-third concentration across
  4 domains.
* `top_third_consistency_count` is integer in `{0, 1, 2, 3, 4}`. Median
  ~1 = most players are concentrated in attacking third on one (rarely
  more) domain at a time.
* Position-dummy families (`cumul_passes_*`, `passes_received_share_*`)
  are mostly zero - one position activates each row.
* `composure_under_pressure_ratio` median ~0.75 means the typical
  player **survives 75% as well under pressure as they pass in
  general**: clear composure drop. Values above 1 (rare) mean the
  player is paradoxically MORE composed when pressed.
* `passes_received_share_above_team_avg` should average ~0 by
  construction (deviation from team mean per fixture x side x cp).
* `shot_to_pass_ratio` and `shot_to_press_ratio` are mostly 0 for
  defenders / GKs and high for attackers - this is the position
  fingerprint we engineered.


## Section B - Bivariate signal vs `scored_after`

In [7]:
"""Merge for analysis."""
m = main[["player_appearance_id", "checkpoint", "scored_after",
          "fixture_id", "position"]].merge(
    cross, on=["player_appearance_id", "checkpoint"], how="left"
)
BASELINE = m["scored_after"].mean()
print(f"baseline scored_after rate: {BASELINE:.4f}")


baseline scored_after rate: 0.0582


In [8]:
"""Pearson r vs target."""
rows = []
for c in NUMERIC:
    s = m[c].dropna()
    if s.std() == 0:
        rows.append({"feature": c, "n": len(s), "pearson_r": float("nan")})
        continue
    t = m.loc[s.index, "scored_after"]
    rows.append({"feature": c, "n": len(s), "pearson_r": float(np.corrcoef(s, t)[0, 1])})
pearson_df = pd.DataFrame(rows).sort_values(
    "pearson_r", key=lambda x: x.abs(), ascending=False
).reset_index(drop=True)
pearson_df


,feature,n,pearson_r
0,passes_received_share_A,3472,0.1330
1,top_third_pass_x_top_third_run,3312,0.1242
2,four_domain_top_third_avg,3482,0.1121
3,top_third_consistency_count,3486,0.1071
4,passes_received_share_D,3482,-0.0902
5,cumul_passes_D,3486,-0.0778
6,passes_received_share_G,3486,-0.0777
7,passes_received_share_above_team_avg,3457,0.0767
8,shot_to_pass_ratio,3431,0.0745
9,cumul_passes_G,3486,-0.0651


In [9]:
"""Binned target rates."""
BINS = {
    "four_domain_top_third_avg":               [-0.01, 0.1, 0.25, 0.5, 1.01],
    "top_third_consistency_count":             [-0.5, 0.5, 1.5, 2.5, 4.5],
    "top_third_pass_x_top_third_run":          [-0.01, 0, 0.05, 0.2, 1.01],
    "cumul_passes_A":                          [-1, 0, 5, 20, 200],
    "cumul_passes_M":                          [-1, 0, 5, 20, 200],
    "cumul_passes_D":                          [-1, 0, 5, 20, 200],
    "cumul_passes_G":                          [-1, 0, 5, 200],
    "passes_received_share_A":                 [-0.01, 0, 0.2, 0.5, 1.01],
    "passes_received_share_M":                 [-0.01, 0, 0.2, 0.5, 1.01],
    "passes_received_share_D":                 [-0.01, 0, 0.2, 0.5, 1.01],
    "passes_received_share_G":                 [-0.01, 0, 1.01],
    "shots_per_pass_received":                 [-0.01, 0, 0.05, 0.2, 1.5],
    "composure_under_pressure_ratio":          [-0.01, 0.5, 1.0, 2.0, 11],
    "last15_passes_received_x_last15_sprints": [-1, 0, 5, 20, 200],
    "passes_received_share_above_team_avg":    [-1, -0.05, 0.05, 1],
    "subbed_x_last15_intensity_shots":         [-0.01, 0, 0.05, 0.5],
    "shot_to_pass_ratio":                      [-0.01, 0, 0.05, 0.2, 5],
    "shot_to_press_ratio":                     [-0.01, 0, 0.05, 0.2, 5],
}

frames = []
for feat, edges in BINS.items():
    t = binned_rate(m, feat, "scored_after", bins=edges)
    t["feature"] = feat
    frames.append(t)
binned_df = pd.concat(frames, ignore_index=True)[
    ["feature", "bin", "n", "n_pos", "rate", "ci_lo", "ci_hi"]
]
binned_df


,feature,bin,n,n_pos,rate,ci_lo,ci_hi
0,four_domain_top_third_avg,"(-0.011, 0.1]",1166,45,0.0386,2.8967e-02,0.0513
1,four_domain_top_third_avg,"(0.1, 0.25]",793,31,0.0391,2.7675e-02,0.0550
2,four_domain_top_third_avg,"(0.25, 0.5]",892,68,0.0762,6.0578e-02,0.0955
3,four_domain_top_third_avg,"(0.5, 1.01]",631,59,0.0935,7.3182e-02,0.1187
4,four_domain_top_third_avg,NaN,4,0,0.0000,0.0000e+00,0.4899
5,top_third_consistency_count,"(-0.501, 0.5]",1398,52,0.0372,2.8477e-02,0.0485
6,top_third_consistency_count,"(0.5, 1.5]",720,33,0.0458,3.2820e-02,0.0637
7,top_third_consistency_count,"(1.5, 2.5]",459,26,0.0566,3.8946e-02,0.0817
8,top_third_consistency_count,"(2.5, 4.5]",909,92,0.1012,8.3251e-02,0.1225
9,top_third_pass_x_top_third_run,"(-0.011, 0.0]",929,54,0.0581,4.4822e-02,0.0751


### B - findings (interpretation)

Patterns to look for:

* **`passes_received_share_A`** should be the strongest single feature -
  it's the per-position version of "target striker"; r should approach
  +0.13 (matching `position_A` from shots EDA).
* **`top_third_pass_x_top_third_run`**, **`four_domain_top_third_avg`**,
  **`top_third_consistency_count`** - all 3 should beat single-domain
  spatial features individually (they aggregate signal).
* **`composure_under_pressure_ratio`** - direction unclear: composed
  attackers (high ratio) might score more, but the median attacker has
  ratio ~0.7 (survives less under press); the relationship may be
  non-monotone.
* **`cumul_passes_A` should be POSITIVE** - whereas pooled `cumul_passes`
  is negative, the position-gated version should isolate "active
  attackers" from "active distributors".
* **`shot_to_pass_ratio` and `shot_to_press_ratio`** - directly
  fingerprint scoring roles; expect strong positive monotone.


## Section C - Redundancy among the 18 columns

In [10]:
"""Pearson correlation matrix."""
corr = cross[NUMERIC].corr(method="pearson")
hi = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .rename("pearson")
        .reset_index()
        .rename(columns={"level_0": "a", "level_1": "b"})
)
hi["abs"] = hi["pearson"].abs()
hi.sort_values("abs", ascending=False).head(15).reset_index(drop=True)


,a,b,pearson,abs
0,four_domain_top_third_avg,top_third_consistency_count,0.9197,0.9197
1,shots_per_pass_received,shot_to_pass_ratio,0.8503,0.8503
2,cumul_passes_G,passes_received_share_G,0.8211,0.8211
3,cumul_passes_D,passes_received_share_D,0.7668,0.7668
4,four_domain_top_third_avg,top_third_pass_x_top_third_run,0.7421,0.7421
5,top_third_consistency_count,top_third_pass_x_top_third_run,0.7294,0.7294
6,shots_per_pass_received,shot_to_press_ratio,0.7015,0.7015
7,shot_to_pass_ratio,shot_to_press_ratio,0.6698,0.6698
8,cumul_passes_M,passes_received_share_M,0.6484,0.6484
9,cumul_passes_A,passes_received_share_A,0.5953,0.5953


### C - findings (interpretation)

Expected near-collinearity:

* `four_domain_top_third_avg` and `top_third_consistency_count` share
  the same 4 parents - high correlation expected.
* `top_third_pass_x_top_third_run` shares `top_third_run_share` parent
  with similar features in `build_cross_features`.
* `passes_received_share_*` and `cumul_passes_*` for the same position
  may correlate because both gate on the same position dummy.

Decision: drop pairs with `|pearson| >= 0.95` keeping the variable with
the higher single-bivariate signal.


## Section D - Cluster-robust significance + BH-FDR

In [11]:
"""Cluster-robust GLM + BH-FDR."""
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests


def cluster_robust_p(df: pd.DataFrame, feature: str) -> tuple[float, float, int]:
    sub = df[[feature, "scored_after", "fixture_id"]].dropna()
    if sub[feature].std() == 0 or len(sub) < 30:
        return float("nan"), float("nan"), len(sub)
    try:
        fitted = smf.logit(f"scored_after ~ {feature}", data=sub).fit(
            disp=False, maxiter=200,
            cov_type="cluster", cov_kwds={"groups": sub["fixture_id"]},
        )
        return float(fitted.params[feature]), float(fitted.pvalues[feature]), len(sub)
    except Exception:
        return float("nan"), float("nan"), len(sub)


rows = []
for f in NUMERIC:
    coef, p, n = cluster_robust_p(m, f)
    r = m[[f, "scored_after"]].corr().iloc[0, 1]
    rows.append({"feature": f, "n": n, "pearson_r": r, "coef": coef, "p_raw": p})
sig = pd.DataFrame(rows)

mask = sig["p_raw"].notna()
adj = np.full(len(sig), np.nan)
if mask.sum():
    _, adj_p, _, _ = multipletests(sig.loc[mask, "p_raw"].values, method="fdr_bh")
    adj[mask.values] = adj_p
sig["p_bh"] = adj
sig["bh_q05"] = sig["p_bh"] < 0.05
sig.sort_values("p_bh").reset_index(drop=True)


C:\Users\tymot\projects\wec\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
C:\Users\tymot\projects\wec\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\tymot\projects\wec\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


,feature,n,pearson_r,coef,p_raw,p_bh,bh_q05
0,cumul_passes_G,3486,-0.0651,-16.2600,1.1263e-107,2.0274e-106,True
1,passes_received_share_G,3486,-0.0777,-84.9134,4.1320e-101,3.7188e-100,True
2,top_third_pass_x_top_third_run,3312,0.1242,2.7946,7.6095e-09,4.5657e-08,True
3,passes_received_share_A,3472,0.1330,1.9297,7.7653e-07,3.4944e-06,True
4,four_domain_top_third_avg,3482,0.1121,1.9994,7.1278e-05,2.5660e-04,True
5,top_third_consistency_count,3486,0.1071,0.3002,2.3546e-04,7.0639e-04,True
6,passes_received_share_above_team_avg,3457,0.0767,3.4071,5.7912e-04,1.4892e-03,True
7,passes_received_share_D,3482,-0.0902,-1.9314,7.2128e-03,1.6229e-02,True
8,shots_per_pass_received,3446,0.0575,1.7061,1.1277e-02,2.0940e-02,True
9,shot_to_pass_ratio,3431,0.0745,1.2850,1.1634e-02,2.0940e-02,True


### D - findings (interpretation)

Three buckets:

1. **Survives BH q=0.05 with rate above baseline -> KEEP.**
2. **BH-significant but rate flat -> drop (perfect-separation
   artefact, typically goalkeeper dummy columns).**
3. **Pearson borderline + non-significant BH -> demote unless
   tree-feature-importance later rescues it.**

Expected survivors: `passes_received_share_A`,
`top_third_pass_x_top_third_run`, `four_domain_top_third_avg`,
`top_third_consistency_count`, `shot_to_pass_ratio`.


## Section E - Final manifest

In [12]:
"""Programmatic manifest."""
def best_non_baseline_bin(df: pd.DataFrame, feature: str) -> tuple[float, float, float, int]:
    sub = df[df["feature"] == feature]
    if sub.empty:
        return (float("nan"),) * 3 + (0,)
    sub = sub.sort_values("rate", ascending=False).iloc[0]
    return float(sub["rate"]), float(sub["ci_lo"]), float(sub["ci_hi"]), int(sub["n"])


rows = []
for feat_name in NUMERIC:
    rate, lo, hi, n = best_non_baseline_bin(binned_df, feat_name)
    rows.append({"feature": feat_name, "best_rate": rate,
                 "ci_lo": lo, "ci_hi": hi, "n": n})

manifest = pd.DataFrame(rows).merge(
    sig[["feature", "pearson_r", "p_bh", "bh_q05"]], on="feature", how="left"
)


def verdict(row) -> str:
    if pd.isna(row["best_rate"]):
        return "drop (no data)"
    rate_above = row["best_rate"] > BASELINE * 1.2
    ci_above = (not pd.isna(row["ci_lo"]) and row["ci_lo"] > BASELINE)
    if row["bh_q05"] is True and (rate_above or ci_above):
        return "KEEP"
    if ci_above:
        return "keep (cautious)"
    if row["best_rate"] > BASELINE * 1.4 and row["n"] >= 30:
        return "keep (cautious)"
    return "drop (flat)"


manifest["verdict"] = manifest.apply(verdict, axis=1)
manifest.sort_values(["verdict", "p_bh"]).reset_index(drop=True)


,feature,best_rate,ci_lo,ci_hi,n,pearson_r,p_bh,bh_q05,verdict
0,top_third_pass_x_top_third_run,0.1237,0.0975,0.1557,493,0.1242,4.5657e-08,True,KEEP
1,passes_received_share_A,0.2143,0.0757,0.4759,14,0.1330,3.4944e-06,True,KEEP
2,four_domain_top_third_avg,0.0935,0.0732,0.1187,631,0.1121,2.5660e-04,True,KEEP
3,top_third_consistency_count,0.1012,0.0833,0.1225,909,0.1071,7.0639e-04,True,KEEP
4,passes_received_share_above_team_avg,0.1111,0.0878,0.1396,567,0.0767,1.4892e-03,True,KEEP
5,passes_received_share_D,0.0737,0.0636,0.0852,2240,-0.0902,1.6229e-02,True,KEEP
6,shots_per_pass_received,0.1353,0.0874,0.2038,133,0.0575,2.0940e-02,True,KEEP
7,shot_to_pass_ratio,0.1117,0.0749,0.1633,197,0.0745,2.0940e-02,True,KEEP
8,cumul_passes_G,0.0641,0.0561,0.0731,3168,-0.0651,2.0274e-106,True,drop (flat)
9,passes_received_share_G,0.0641,0.0560,0.0731,3169,-0.0777,3.7188e-100,True,drop (flat)


### Closing notes

* This manifest is the deliverable. Compare against `eda_cross_features.ipynb`
  and `eda_press_cross_features.ipynb` to assemble the final feature
  set for the modelling step.
* For RQ4 ("does pass + pressure data help?"), the survivors here are
  the strongest evidence that 4-table joint features beat any
  single-domain or 2-table cross engineered earlier.
* Cross features that fail BH may still contribute via tree feature
  importance during modelling - especially `composure_under_pressure_ratio`
  and `last15_passes_received_x_last15_sprints` which carry compound
  mechanisms a linear logit struggles to detect.
